# burnscars thin Colab runner

Zero logic in cells: every cell calls `burnscars.colab` or a tool.

**Colab compute is shared with other projects. Run GPU cells ONLY for an allocation the user approved in-session, recorded as APPROVED in `docs/COMPUTE_LEDGER.md` at the cloned commit.** GPU-tier tools refuse otherwise, and `start_allocation` sets a hard deadline.

Secrets: `GH_TOKEN` lives in the Colab secrets panel only.

In [ ]:
# 1. Mount Drive (durable storage; never the working directory)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Auth BEFORE clone (token from the secrets panel, never pasted)
import os
from google.colab import userdata
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')

In [ ]:
# 3. Clone or update. Re-run after pushing a fix; same kernel.
!test -d /content/Burn-Scars || git clone https://$GH_TOKEN@github.com/meteorboyF/Burn-Scars.git /content/Burn-Scars
!cd /content/Burn-Scars && git fetch -q origin && git reset -q --hard origin/main

In [ ]:
# 4. Environment: local caches/outputs, fingerprint-gated install, printed SHA + GPU
import sys; sys.path.insert(0, '/content/Burn-Scars/src')
import importlib, burnscars.colab as bc; importlib.reload(bc)
bc.configure_env(); bc.ensure_requirements('requirements-colab.txt'); bc.print_sha()
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 5. Data to LOCAL disk (Drive tarball cache, SHA-verified)
bc.restore_data()

## Pilot P-01 (allocation A-001) — run only after APPROVED

In [ ]:
# 6. Arm the approved allocation (hard deadline)
bc.start_allocation('A-001', gpu_hours=0.0)  # set to the approved GPU-hours

In [ ]:
# 7. Tuning (random regime, inner fold 0 of outer fold 0)
for lr in ['0.0003', '0.001', '0.003']:
    bc.run('train_seg', config=f'configs/grid/pilot_v1/tune_lr{lr}.yaml'); bc.copy_to_drive(f'grid/pilot_v1/tune_lr{lr}')

In [ ]:
# 8. Select lr by the pre-registered rule, generate outer configs (commit them from the laptop afterwards)
!cd /content/Burn-Scars && PYTHONPATH=src python scripts/make_grid_configs.py --grid pilot_v1 --stage outer

In [ ]:
# 9. Outer runs: 5 random + 5 grouped + 1 as-published
import glob, os
for c in sorted(glob.glob('/content/Burn-Scars/configs/grid/pilot_v1/unet_*.yaml')):
    n = os.path.basename(c)[:-5]; bc.run('train_seg', config=c); bc.copy_to_drive(f'grid/pilot_v1/{n}')